In [1]:
library(keras)
library(tensorflow)
library(caret)
library(magrittr)
library(abind)

# Configuration - IMPROVED
CONFIG <- list(
  base_dir = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train",
  classes = c("Chickenpox", "Cowpox", "HFMD", "Healthy", "Measles", "Monkeypox"),
  img_size = c(128, 128),
  batch_size = 32,
  max_pairs_per_class = 200,  # Increased
  learning_rate = 0.001,      # Slightly higher
  epochs = 50,
  max_per_class = 200         # Use much more data!
)

# Load sample of images - IMPROVED to use more data
load_sample_images <- function(base_dir, classes, target_size, max_per_class = 200) {
  images <- list()
  labels <- c()
  
  for (i in seq_along(classes)) {
    class_dir <- file.path(base_dir, classes[i])
    
    if (!dir.exists(class_dir)) {
      cat("Warning: Directory", class_dir, "does not exist\n")
      next
    }
    
    img_files <- list.files(class_dir, pattern = "\\.(jpg|jpeg|png)$", 
                           full.names = TRUE, ignore.case = TRUE)
    
    cat("Found", length(img_files), "images in", classes[i], "\n")
    
    # Use more images if available
    if (length(img_files) > max_per_class) {
      img_files <- sample(img_files, max_per_class)
    }
    
    cat("Loading", length(img_files), "images from", classes[i], "\n")
    
    for (img_file in img_files) {
      tryCatch({
        img <- image_load(img_file, target_size = target_size) %>%
          image_to_array() %>%
          `/`(255)
        
        # Ensure image has correct dimensions (height, width, channels)
        if (length(dim(img)) == 2) {
          # Grayscale image - add channel dimension
          img <- array(img, dim = c(dim(img), 1))
        }
        
        # Convert grayscale to RGB if needed
        if (dim(img)[3] == 1) {
          img <- array(rep(img, 3), dim = c(dim(img)[1:2], 3))
        }
        
        images <- append(images, list(img))
        labels <- c(labels, i)
      }, error = function(e) {
        cat("Error loading", img_file, ":", e$message, "\n")
      })
    }
  }
  
  if (length(images) == 0) {
    stop("No images were loaded successfully")
  }
  
  # Convert list of 3D arrays to 4D array properly
  first_img_dims <- dim(images[[1]])
  n_images <- length(images)
  image_array <- array(0, dim = c(n_images, first_img_dims[1], first_img_dims[2], first_img_dims[3]))
  
  # Fill the array
  for (i in seq_len(n_images)) {
    image_array[i, , , ] <- images[[i]]
  }
  
  cat("Final image array dimensions:", dim(image_array), "\n")
  cat("Class distribution:", table(labels), "\n")
  
  return(list(images = image_array, labels = labels))
}

# IMPROVED: Better pair creation strategy
create_smart_pairs <- function(images, labels, max_pairs_per_class = 200) {
  n_classes <- length(unique(labels))
  class_indices <- split(seq_along(labels), labels)
  
  pairs_left <- list()
  pairs_right <- list()
  pair_labels <- c()
  
  # Check dimensions of images array
  img_dims <- dim(images)
  cat("Images array dimensions:", img_dims, "\n")
  
  if (length(img_dims) != 4) {
    stop("Expected 4D image array (batch, height, width, channels), got ", length(img_dims), "D")
  }
  
  # Create positive pairs (same class) - more diverse sampling
  positive_pairs <- 0
  for (class_idx in seq_len(n_classes)) {
    indices <- class_indices[[class_idx]]
    class_name <- names(class_indices)[class_idx]
    
    if (length(indices) < 2) {
      cat("Skipping class", class_idx, "- not enough samples\n")
      next
    }
    
    # Shuffle indices for better diversity
    indices <- sample(indices)
    
    # Create pairs more systematically
    pairs_created <- 0
    max_attempts <- min(length(indices) * (length(indices) - 1) / 2, max_pairs_per_class)
    
    for (i in seq_len(length(indices) - 1)) {
      if (pairs_created >= max_pairs_per_class) break
      
      for (j in (i + 1):length(indices)) {
        if (pairs_created >= max_pairs_per_class) break
        
        img1 <- images[indices[i], , , , drop = FALSE]
        img2 <- images[indices[j], , , , drop = FALSE]
        
        pairs_left <- append(pairs_left, list(img1))
        pairs_right <- append(pairs_right, list(img2))
        pair_labels <- c(pair_labels, 1)
        
        pairs_created <- pairs_created + 1
        positive_pairs <- positive_pairs + 1
      }
    }
    
    cat("Created", pairs_created, "positive pairs for class", class_idx, "\n")
  }
  
  # Create equal number of negative pairs (different classes)
  negative_pairs <- 0
  target_negative_pairs <- positive_pairs
  
  cat("Creating", target_negative_pairs, "negative pairs...\n")
  
  # More systematic negative pair creation
  attempts <- 0
  max_attempts <- target_negative_pairs * 3  # Avoid infinite loop
  
  while (negative_pairs < target_negative_pairs && attempts < max_attempts) {
    attempts <- attempts + 1
    
    # Select two different classes
    classes_to_sample <- sample(seq_len(n_classes), 2)
    class1 <- classes_to_sample[1]
    class2 <- classes_to_sample[2]
    
    indices1 <- class_indices[[class1]]
    indices2 <- class_indices[[class2]]
    
    if (length(indices1) > 0 && length(indices2) > 0) {
      idx1 <- sample(indices1, 1)
      idx2 <- sample(indices2, 1)
      
      img1 <- images[idx1, , , , drop = FALSE]
      img2 <- images[idx2, , , , drop = FALSE]
      
      pairs_left <- append(pairs_left, list(img1))
      pairs_right <- append(pairs_right, list(img2))
      pair_labels <- c(pair_labels, 0)
      
      negative_pairs <- negative_pairs + 1
    }
  }
  
  cat("Created", positive_pairs, "positive pairs and", negative_pairs, "negative pairs\n")
  
  # Convert to arrays
  n_pairs <- length(pairs_left)
  if (n_pairs == 0) {
    stop("No pairs were created")
  }
  
  # Get dimensions from first pair
  pair_dims <- dim(pairs_left[[1]])
  
  # Create arrays
  left_array <- array(0, dim = c(n_pairs, pair_dims[2], pair_dims[3], pair_dims[4]))
  right_array <- array(0, dim = c(n_pairs, pair_dims[2], pair_dims[3], pair_dims[4]))
  
  # Fill arrays
  for (i in seq_len(n_pairs)) {
    left_array[i, , , ] <- pairs_left[[i]][1, , , ]
    right_array[i, , , ] <- pairs_right[[i]][1, , , ]
  }
  
  cat("Final arrays - Left:", dim(left_array), "Right:", dim(right_array), "\n")
  
  return(list(
    left = left_array,
    right = right_array,
    labels = pair_labels
  ))
}

# IMPROVED: Lighter but more effective base network
create_base_network <- function(input_shape) {
  input <- layer_input(shape = input_shape)
  
  x <- input %>%
    # Block 1
    layer_conv_2d(64, c(3, 3), activation = "relu", padding = "same") %>%
    layer_batch_normalization() %>%
    layer_max_pooling_2d(c(2, 2)) %>%
    layer_dropout(0.25) %>%
    
    # Block 2
    layer_conv_2d(128, c(3, 3), activation = "relu", padding = "same") %>%
    layer_batch_normalization() %>%
    layer_max_pooling_2d(c(2, 2)) %>%
    layer_dropout(0.25) %>%
    
    # Block 3
    layer_conv_2d(256, c(3, 3), activation = "relu", padding = "same") %>%
    layer_batch_normalization() %>%
    layer_max_pooling_2d(c(2, 2)) %>%
    layer_dropout(0.25) %>%
    
    # Block 4
    layer_conv_2d(512, c(3, 3), activation = "relu", padding = "same") %>%
    layer_batch_normalization() %>%
    layer_max_pooling_2d(c(2, 2)) %>%
    layer_dropout(0.25) %>%
    
    # Dense layers
    layer_flatten() %>%
    layer_dense(1024, activation = "relu") %>%
    layer_batch_normalization() %>%
    layer_dropout(0.5) %>%
    layer_dense(512, activation = "relu") %>%
    layer_batch_normalization() %>%
    layer_dropout(0.5) %>%
    layer_dense(256, activation = NULL)  # Embedding layer
  
  return(keras_model(input, x))
}

# SIMPLIFIED: Use binary crossentropy instead of contrastive loss to avoid tensor issues
binary_accuracy <- function(y_true, y_pred) {
  threshold <- 0.5
  y_pred_binary <- k_cast(y_pred > threshold, "float32")
  return(k_mean(k_equal(y_true, y_pred_binary)))
}

# Euclidean distance function for use outside of the model
euclidean_distance_np <- function(a, b) {
  sqrt(sum((a - b)^2))
}

# Main training function - SIMPLIFIED APPROACH
train_siamese_network <- function() {
  cat("Starting Siamese Network Training\n")
  cat("=================================\n")
  
  # Load data
  cat("Loading images...\n")
  data <- load_sample_images(CONFIG$base_dir, CONFIG$classes, CONFIG$img_size, CONFIG$max_per_class)
  cat("Loaded", dim(data$images)[1], "images total\n")
  
  # Create pairs
  cat("Creating pairs...\n")
  pairs <- create_smart_pairs(data$images, data$labels, CONFIG$max_pairs_per_class)
  cat("Created", length(pairs$labels), "total pairs\n")
  
  # Shuffle pairs
  set.seed(42)
  n_pairs <- length(pairs$labels)
  shuffle_idx <- sample(seq_len(n_pairs))
  
  pairs$left <- pairs$left[shuffle_idx, , , , drop = FALSE]
  pairs$right <- pairs$right[shuffle_idx, , , , drop = FALSE]
  pairs$labels <- pairs$labels[shuffle_idx]
  
  # Split data
  train_idx <- seq_len(floor(0.8 * n_pairs))
  val_idx <- (length(train_idx) + 1):n_pairs
  
  train_left <- pairs$left[train_idx, , , , drop = FALSE]
  train_right <- pairs$right[train_idx, , , , drop = FALSE]
  train_labels <- pairs$labels[train_idx]
  
  val_left <- pairs$left[val_idx, , , , drop = FALSE]
  val_right <- pairs$right[val_idx, , , , drop = FALSE]
  val_labels <- pairs$labels[val_idx]
  
  cat("Training pairs:", length(train_labels), "(positive:", sum(train_labels), "negative:", sum(train_labels == 0), ")\n")
  cat("Validation pairs:", length(val_labels), "(positive:", sum(val_labels), "negative:", sum(val_labels == 0), ")\n")
  
  # Build model - SIMPLIFIED VERSION
  cat("Building model...\n")
  input_shape <- c(CONFIG$img_size, 3)
  base_network <- create_base_network(input_shape)
  
  # Input layers
  input_a <- layer_input(shape = input_shape, name = "input_a")
  input_b <- layer_input(shape = input_shape, name = "input_b")
  
  # Process inputs through base network
  processed_a <- base_network(input_a)
  processed_b <- base_network(input_b)
  
  # Combine the features and use a simple dense layer for similarity
  # This avoids the tensor shape issues with lambda layers
  combined <- layer_concatenate(list(processed_a, processed_b))
  
  # Add layers to learn similarity
  similarity <- combined %>%
    layer_dense(512, activation = "relu") %>%
    layer_dropout(0.5) %>%
    layer_dense(256, activation = "relu") %>%
    layer_dropout(0.5) %>%
    layer_dense(1, activation = "sigmoid")  # Output probability of being the same class
  
  # Create model
  model <- keras_model(list(input_a, input_b), similarity)
  
  # Compile model with binary crossentropy
  model %>% compile(
    optimizer = optimizer_adam(learning_rate = CONFIG$learning_rate),
    loss = "binary_crossentropy",
    metrics = list("accuracy")
  )
  
  # Print model summary
  cat("Model summary:\n")
  summary(model)
  
  # Callbacks
  callbacks <- list(
    callback_early_stopping(
      monitor = "val_loss",
      patience = 15, 
      restore_best_weights = TRUE, 
      verbose = 1
    ),
    callback_reduce_lr_on_plateau(
      monitor = "val_loss",
      patience = 8, 
      factor = 0.5, 
      verbose = 1, 
      min_lr = 1e-7
    )
  )
  
  # Train model
  cat("Starting training...\n")
  history <- model %>% fit(
    x = list(train_left, train_right),
    y = train_labels,
    validation_data = list(list(val_left, val_right), val_labels),
    batch_size = CONFIG$batch_size,
    epochs = CONFIG$epochs,
    callbacks = callbacks,
    verbose = 1
  )
  
  # Evaluate model
  cat("Evaluating model...\n")
  val_predictions <- predict(model, list(val_left, val_right), verbose = 0)
  
  # Convert to binary predictions
  final_predictions <- ifelse(val_predictions > 0.5, 1, 0)
  final_accuracy <- mean(final_predictions == val_labels)
  
  cat("Validation Accuracy:", round(final_accuracy * 100, 2), "%\n")
  
  # Print detailed statistics
  pos_probs <- val_predictions[val_labels == 1]
  neg_probs <- val_predictions[val_labels == 0]
  
  cat("Probability statistics:\n")
  cat("Positive pairs - Mean:", round(mean(pos_probs), 3), 
      "Std:", round(sd(pos_probs), 3), 
      "Range:", round(range(pos_probs), 3), "\n")
  cat("Negative pairs - Mean:", round(mean(neg_probs), 3), 
      "Std:", round(sd(neg_probs), 3), 
      "Range:", round(range(neg_probs), 3), "\n")
  
  # Separation measure
  separation <- abs(mean(pos_probs) - mean(neg_probs))
  cat("Probability separation:", round(separation, 3), "\n")
  
  # Create feature extractor
  feature_extractor <- keras_model(inputs = base_network$input, 
                                  outputs = base_network$output)
  
  return(list(
    model = model,
    feature_extractor = feature_extractor,
    history = history,
    base_network = base_network,
    threshold = 0.5
  ))
}

# Check directory and run training
cat("Configuration:\n")
str(CONFIG)
cat("\n")

if (!dir.exists(CONFIG$base_dir)) {
  cat("Error: Base directory does not exist:", CONFIG$base_dir, "\n")
  cat("Please update CONFIG$base_dir with the correct path\n")
} else {
  cat("Directory exists. Starting training...\n")
  results <- train_siamese_network()
  cat("Training completed successfully!\n")
}

Loading required package: ggplot2

Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:tensorflow’:

    train


The following object is masked from ‘package:httr’:

    progress




Configuration:
List of 8
 $ base_dir           : chr "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train"
 $ classes            : chr [1:6] "Chickenpox" "Cowpox" "HFMD" "Healthy" ...
 $ img_size           : num [1:2] 128 128
 $ batch_size         : num 32
 $ max_pairs_per_class: num 200
 $ learning_rate      : num 0.001
 $ epochs             : num 50
 $ max_per_class      : num 200

Directory exists. Starting training...
Starting Siamese Network Training
Loading images...
Found 700 images in Chickenpox 
Loading 200 images from Chickenpox 
Found 686 images in Cowpox 
Loading 200 images from Cowpox 
Found 1624 images in HFMD 
Loading 200 images from HFMD 
Found 1162 images in Healthy 
Loading 200 images from Healthy 
Found 518 images in Measles 
Loading 200 images from Measles 
Found 2828 images in Monkeypox 
Loading 200 images from Monkeypox 
Final image array dimensions: 1200 128 128 3 
Class distribution: 200 200 200 